In [ ]:
import json
import os
import time
import string
import re
from collections import Counter

import numpy as np
import torch
import matplotlib.pyplot as plt
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
    DataCollatorForSeq2Seq,
)
import evaluate
from sentence_transformers import SentenceTransformer, util


os.environ["TOKENIZERS_PARALLELISM"] = "false"


# ==========================================
# 1. Load and Prepare Data
# ==========================================

json_file = "version1_2_fixed_other.json"   # Change to your file

with open(json_file, "r", encoding="utf-8") as f:
    data = json.load(f)["data"]


flat_data = []
sample_id = 0

for item in data:
    title = item.get("title", "") or ""

    for para in item.get("paragraphs", []):
        context = para.get("context", "") or ""

        for qa in para.get("qas", []):

            # Skip questions without answers
            if not qa.get("answers"):
                continue

            answer = qa["answers"][0] or {}

            flat_data.append({
                "id": qa.get("id", f"sample_{sample_id}"),
                "title": title,
                "context": context,
                "question": qa.get("question", "") or "",
                "answer_text": answer.get("text", "") or "",
                "answer_start": answer.get("answer_start", -1),
                "answer_end": answer.get("answer_end", -1),
            })

            sample_id += 1


from sklearn.model_selection import train_test_split

# Split the data: 80% train, 20% test
train_raw, test_raw = train_test_split(flat_data, test_size=0.2, random_state=42)

train_dataset = Dataset.from_list(train_raw)
test_dataset = Dataset.from_list(test_raw)

print(f"✅ Loaded {len(flat_data)} total samples")
print(f"✅ Training samples: {len(train_dataset)}")
print(f"✅ Testing samples: {len(test_dataset)}")

if len(train_dataset) > 0:
    print(train_dataset[0])


# ==========================================
# 2. Tokenization and Preprocessing
# ==========================================

model_checkpoint = "t5-small"  # Can also use "t5-base" for better performance
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)


def preprocess_function(examples):
    """
    T5 uses a prefix-style format for QA tasks.

    Input:
        "question: <question> context: <context>"

    Target:
        "<answer>"
    """
    inputs = []
    targets = []

    for question, context, answer in zip(
        examples["question"],
        examples["context"],
        examples["answer_text"]
    ):
        input_text = f"question: {question} context: {context}"
        inputs.append(input_text)
        targets.append(answer)

    # Tokenize inputs
    model_inputs = tokenizer(
        inputs,
        max_length=512,
        truncation=True,
        padding="max_length",
    )

    # Tokenize targets / labels
    labels = tokenizer(
        targets,
        max_length=64,
        truncation=True,
        padding="max_length",
    )

    # Replace padding token ids in labels with -100.
    # This makes the loss ignore padding tokens.
    labels_with_ignore_index = []
    for label_seq in labels["input_ids"]:
        labels_with_ignore_index.append(
            [
                -100 if token_id == tokenizer.pad_token_id else token_id
                for token_id in label_seq
            ]
        )

    model_inputs["labels"] = labels_with_ignore_index

    # Keep these for final evaluation
    model_inputs["example_id"] = examples["id"]
    model_inputs["answer_texts"] = examples["answer_text"]

    return model_inputs


# Remove all original columns after mapping.
# preprocess_function returns the new columns needed for training/evaluation.
# Define columns to remove BEFORE mapping
cols_to_remove_train = train_dataset.column_names
cols_to_remove_test = test_dataset.column_names

# Tokenize train and test SEPARATELY
tokenized_train = train_dataset.map(
    preprocess_function,
    batched=True,
    remove_columns=cols_to_remove_train,
)

tokenized_test = test_dataset.map(
    preprocess_function,
    batched=True,
    remove_columns=cols_to_remove_test,
)

# For Trainer, keep only tensor-compatible/model columns.
trainer_columns = ["input_ids", "attention_mask", "labels"]

trainer_train_dataset = tokenized_train.remove_columns(
    [col for col in tokenized_train.column_names if col not in trainer_columns]
)

trainer_test_dataset = tokenized_test.remove_columns(
    [col for col in tokenized_test.column_names if col not in trainer_columns]
)

# ==========================================
# 3. Model Initialization & Training
# ==========================================

model = AutoModelForSeq2SeqLM.from_pretrained(model_checkpoint)

start_train_time = time.time()


training_args = Seq2SeqTrainingArguments(
    output_dir="./results_t5",
    eval_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    num_train_epochs=1,
    weight_decay=0.01,
    report_to="none",
    use_cpu=True,
    logging_steps=1,

    # Your custom evaluation below already uses generation.
    # You can keep this True, but it makes trainer evaluation slower.
    predict_with_generate=True,
)


data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
)


trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=trainer_train_dataset,   # ✅ Train on training set
    eval_dataset=trainer_test_dataset,     # ✅ Validate on unseen test set
    data_collator=data_collator,
    tokenizer=tokenizer,
)


print("Starting Training...")
trainer.train()

end_train_time = time.time()
training_time = end_train_time - start_train_time

print(f"Training completed in {training_time:.2f} seconds.")


# ==========================================
# 3.5 Plot Training Loss
# ==========================================

log_history = trainer.state.log_history

steps = []
losses = []

for log in log_history:
    if "loss" in log:
        steps.append(log["step"])
        losses.append(log["loss"])


if steps:
    plt.figure(figsize=(10, 5))
    plt.plot(steps, losses, linewidth=2)
    plt.title("Training Loss over Steps (T5)")
    plt.xlabel("Step")
    plt.ylabel("Loss")
    plt.grid(True, alpha=0.3)
    plt.tight_layout()

    plt.savefig("training_loss_t5.png", dpi=300, bbox_inches="tight")
    plt.show()
    plt.close()
else:
    print("No loss history found to plot.")


# ==========================================
# 4. Comprehensive Evaluation Metrics
# ==========================================

print("Evaluating Model...")


def normalize_text(s):
    def remove_articles(text):
        return re.sub(r"\b(a|an|the)\b", " ", text)

    def white_space_fix(text):
        return " ".join(text.split())

    def remove_punc(text):
        exclude = set(string.punctuation)
        return "".join(ch for ch in text if ch not in exclude)

    def lower(text):
        return text.lower()

    return white_space_fix(remove_articles(remove_punc(lower(s))))


def compute_qa_f1(prediction, ground_truth):
    prediction_tokens = normalize_text(prediction).split()
    ground_truth_tokens = normalize_text(ground_truth).split()

    common = Counter(prediction_tokens) & Counter(ground_truth_tokens)
    num_same = sum(common.values())

    if num_same == 0:
        return 0.0

    precision = 1.0 * num_same / len(prediction_tokens)
    recall = 1.0 * num_same / len(ground_truth_tokens)

    if precision + recall == 0:
        return 0.0

    # FIXED: denominator should be precision + recall
    return 2 * (precision * recall) / (precision + recall)


def compute_qa_exact_match(prediction, ground_truth):
    return 1.0 if normalize_text(prediction) == normalize_text(ground_truth) else 0.0


bleu_metric = evaluate.load("bleu")
rouge_metric = evaluate.load("rouge")
meteor_metric = evaluate.load("meteor")

similarity_model = SentenceTransformer("all-MiniLM-L6-v2", device="cpu")


def evaluate_model(model, dataset, tokenizer):
    device = torch.device("cpu")
    model.to(device)
    model.eval()

    all_preds_text = []
    all_refs_text = []
    confidences = []
    similarities = []

    with torch.no_grad():
        for i in range(len(dataset)):
            sample = dataset[i]

            # Prepare input for T5
            valid_keys = ["input_ids", "attention_mask"]

            inputs = {
                k: torch.tensor(sample[k]).unsqueeze(0).to(device)
                for k in valid_keys
                if k in sample
            }

            if not inputs:
                continue

            # Generate answer using T5
            outputs = model.generate(
                **inputs,

                # FIXED:
                # Use max_new_tokens instead of max_length.
                # Your input sequences are already padded to 512 tokens,
                # so max_length=64 would conflict with the input length.
                max_new_tokens=64,

                num_beams=4,
                early_stopping=True,
                output_scores=True,
                return_dict_in_generate=True,
            )

            # Get generated tokens
            generated_ids = outputs.sequences
            pred_text = tokenizer.decode(
                generated_ids[0],
                skip_special_tokens=True
            ).strip()

            # Calculate confidence from transition scores
            if hasattr(outputs, "scores") and outputs.scores:
                try:
                    # FIXED:
                    # When using num_beams > 1, compute_transition_scores
                    # needs beam_indices to correctly trace the selected beam.
                    transition_scores = model.compute_transition_scores(
                        outputs.sequences,
                        outputs.scores,
                        beam_indices=getattr(outputs, "beam_indices", None),
                        normalize_logits=True,
                    )

                    if transition_scores.numel() == 0:
                        confidence = 0.5
                    else:
                        confidence = float(torch.exp(transition_scores[0]).mean().item())

                        if not np.isfinite(confidence):
                            confidence = 0.5

                except Exception:
                    confidence = 0.5
            else:
                confidence = 0.5

            ref_text = sample.get("answer_texts", "")

            all_preds_text.append(pred_text)
            all_refs_text.append(ref_text)
            confidences.append(confidence)

            # Semantic similarity
            if not pred_text:
                sim_score = 0.0
            else:
                pred_emb = similarity_model.encode(
                    pred_text,
                    convert_to_tensor=True,
                    device="cpu"
                )
                ref_emb = similarity_model.encode(
                    ref_text,
                    convert_to_tensor=True,
                    device="cpu"
                )
                sim_score = float(util.cos_sim(pred_emb, ref_emb).item())

            similarities.append(sim_score)

    # If no samples were evaluated
    if not all_preds_text:
        return {
            "F1 Score (Token Overlap)": 0.0,
            "Accuracy (Exact Match)": 0.0,
            "BLEU": 0.0,
            "ROUGE-L": 0.0,
            "METEOR": 0.0,
            "Avg Confidence": 0.0,
            "Avg Semantic Similarity": 0.0,
            "Training Time (seconds)": training_time,
        }

    f1_scores = [
        compute_qa_f1(pred, ref)
        for pred, ref in zip(all_preds_text, all_refs_text)
    ]

    em_scores = [
        compute_qa_exact_match(pred, ref)
        for pred, ref in zip(all_preds_text, all_refs_text)
    ]

    avg_f1 = sum(f1_scores) / len(f1_scores) if f1_scores else 0.0
    avg_em = sum(em_scores) / len(em_scores) if em_scores else 0.0

    bleu_score = bleu_metric.compute(
        predictions=all_preds_text,
        references=[[ref] for ref in all_refs_text],
    )["bleu"]

    rouge_score = rouge_metric.compute(
        predictions=all_preds_text,
        references=all_refs_text,
        rouge_types=["rougeL"],
    )["rougeL"]

    meteor_score = meteor_metric.compute(
        predictions=all_preds_text,
        references=all_refs_text,
    )["meteor"]

    avg_confidence = float(np.mean(confidences)) if confidences else 0.0
    avg_similarity = float(np.mean(similarities)) if similarities else 0.0

    return {
        "F1 Score (Token Overlap)": avg_f1,
        "Accuracy (Exact Match)": avg_em,
        "BLEU": bleu_score,
        "ROUGE-L": rouge_score,
        "METEOR": meteor_score,
        "Avg Confidence": avg_confidence,
        "Avg Semantic Similarity": avg_similarity,
        "Training Time (seconds)": training_time,
    }


# ✅ Evaluate ONLY on the unseen test set (keeps answer_texts for metrics)
results = evaluate_model(model, tokenized_test, tokenizer)

print("\n" + "=" * 45)
print("FINAL EVALUATION RESULTS (T5)")
print("=" * 45)

for metric, value in results.items():
    if isinstance(value, float):
        print(f"{metric:<28}: {value:.4f}")
    else:
        print(f"{metric:<28}: {value}")

In [ ]:
import json
import time
import torch
import numpy as np
import matplotlib.pyplot as plt
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
    DataCollatorForSeq2Seq
)
# NEW: Import PEFT libraries for LoRA
from peft import LoraConfig, get_peft_model, TaskType
import evaluate
from sentence_transformers import SentenceTransformer, util
import re
import string
from collections import Counter

from sklearn.model_selection import train_test_split

# ==========================================
# 1. Load and Prepare Data
# ==========================================

json_file = "version1_2_fixed_other.json"   # Change to your file

with open(json_file, "r", encoding="utf-8") as f:
    data = json.load(f)["data"]

flat_data = []
sample_id = 0

for item in data:
    title = item.get("title", "")

    for para in item.get("paragraphs", []):
        context = para.get("context", "")

        for qa in para.get("qas", []):

            # Skip questions without answers
            if not qa.get("answers"):
                continue

            answer = qa["answers"][0]

            flat_data.append({
                "id": qa.get("id", f"sample_{sample_id}"),
                "title": title,
                "context": context,
                "question": qa.get("question", ""),
                "answer_text": answer.get("text", ""),
                "answer_start": answer.get("answer_start", -1),
                "answer_end": answer.get("answer_end", -1)
            })

            sample_id += 1

train_raw, test_raw = train_test_split(flat_data, test_size=0.2, random_state=42)

train_dataset = Dataset.from_list(train_raw)
test_dataset = Dataset.from_list(test_raw)

print(f"✅ Training samples: {len(train_dataset)}")
print(f"✅ Testing samples: {len(test_dataset)}")
# print(f"Loaded {len(dataset)} samples")
# print(dataset[0])

# ==========================================
# 2. Tokenization and Preprocessing
# ==========================================

# CHANGED: Use T5 model
model_checkpoint = "t5-small"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)


def preprocess_function(examples):
    """
    T5 uses a prefix-style format for QA tasks.
    Input: "question: <question> context: <context>"
    Target: "<answer>"
    """
    inputs = []
    targets = []

    for question, context, answer in zip(
        examples["question"],
        examples["context"],
        examples["answer_text"]
    ):
        # Format input with T5-style prefix
        input_text = f"question: {question} context: {context}"
        inputs.append(input_text)
        targets.append(answer)

    # Tokenize inputs
    model_inputs = tokenizer(
        inputs,
        max_length=512,
        truncation=True,
        padding="max_length"
    )

    # Tokenize targets (labels)
    labels = tokenizer(
        targets,
        max_length=64,
        truncation=True,
        padding="max_length"
    )

    model_inputs["labels"] = labels["input_ids"]
    model_inputs["example_id"] = examples["id"]
    model_inputs["answer_texts"] = examples["answer_text"]

    return model_inputs


# Remove unnecessary columns
cols_to_remove = ["question", "context", "answer_text", "answer_start", "answer_end", "title"]

# Tokenize train and test SEPARATELY
tokenized_train = train_dataset.map(preprocess_function, batched=True, remove_columns=cols_to_remove)
tokenized_test = test_dataset.map(preprocess_function, batched=True, remove_columns=cols_to_remove)

# ==========================================
# 3. Model Initialization & LoRA Configuration
# ==========================================

# Load base model
base_model = AutoModelForSeq2SeqLM.from_pretrained(model_checkpoint)

# Configure LoRA
lora_config = LoraConfig(
    r=8,  # Rank of the update matrices
    lora_alpha=32,  # Scaling factor
    target_modules=["q", "v"],  # Modules to apply LoRA to (query and value projections)
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.SEQ_2_SEQ_LM  # Specific task type for T5
)

# Get PeftModel
model = get_peft_model(base_model, lora_config)

# Print trainable parameters to verify LoRA is active
model.print_trainable_parameters()

start_train_time = time.time()

# Training Arguments
training_args = Seq2SeqTrainingArguments(
    output_dir="./results_t5_lora",
    eval_strategy="epoch",
    learning_rate=1e-4,  # LoRA usually allows slightly higher LR than full fine-tuning
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    num_train_epochs=1,  # Increased epochs since LoRA trains faster/converges differently
    weight_decay=0.01,
    report_to="none",
    use_cpu=True,
    logging_steps=1,
    predict_with_generate=True,
    fp16=False,  # Ensure False for CPU
)

# Data collator for seq2seq models
data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,   # ✅ Train on 1600
    eval_dataset=tokenized_test,     # ✅ Validate on 400 unseen
    data_collator=data_collator,
)
print("Starting Training with LoRA...")
trainer.train()

end_train_time = time.time()
training_time = end_train_time - start_train_time
print(f"Training completed in {training_time:.2f} seconds.")

# ==========================================
# 3.5 Plot Training Loss
# ==========================================
log_history = trainer.state.log_history
steps = []
losses = []

for log in log_history:
    if "loss" in log:
        steps.append(log["step"])
        losses.append(log["loss"])

if steps:
    plt.figure(figsize=(10, 5))
    plt.plot(steps, losses, linewidth=2)
    plt.title("Training Loss over Steps (T5 + LoRA)")
    plt.xlabel("Step")
    plt.ylabel("Loss")
    plt.grid(True, alpha=0.3)
    plt.tight_layout()

    plt.savefig("training_loss_t5_lora.png", dpi=300, bbox_inches="tight")
    plt.show()
    plt.close()
else:
    print("No loss history found to plot.")

# ==========================================
# 4. Comprehensive Evaluation Metrics (String-Based QA)
# ==========================================
print("Evaluating Model...")


def normalize_text(s):
    def remove_articles(text):
        return re.sub(r'\b(a|an|the)\b', ' ', text)

    def white_space_fix(text):
        return ' '.join(text.split())

    def remove_punc(text):
        exclude = set(string.punctuation)
        return ''.join(ch for ch in text if ch not in exclude)

    def lower(text):
        return text.lower()

    return white_space_fix(remove_articles(remove_punc(lower(s))))


def compute_qa_f1(prediction, ground_truth):
    prediction_tokens = normalize_text(prediction).split()
    ground_truth_tokens = normalize_text(ground_truth).split()
    common = Counter(prediction_tokens) & Counter(ground_truth_tokens)
    num_same = sum(common.values())
    if num_same == 0:
        return 0.0
    precision = 1.0 * num_same / len(prediction_tokens)
    recall = 1.0 * num_same / len(ground_truth_tokens)
    f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0.0
    return f1


def compute_qa_exact_match(prediction, ground_truth):
    return 1.0 if normalize_text(prediction) == normalize_text(ground_truth) else 0.0


bleu_metric = evaluate.load("bleu")
rouge_metric = evaluate.load("rouge")
meteor_metric = evaluate.load("meteor")

similarity_model = SentenceTransformer('all-MiniLM-L6-v2', device='cpu')


def evaluate_model(model, dataset, tokenizer):
    device = torch.device("cpu")
    model.to(device)
    model.eval()

    all_preds_text = []
    all_refs_text = []
    confidences = []
    similarities = []

    with torch.no_grad():
        for i in range(len(dataset)):
            sample = dataset[i]

            # Prepare input for T5
            valid_keys = ['input_ids', 'attention_mask']

            inputs = {
                k: torch.tensor(v).unsqueeze(0).to(device)
                for k, v in sample.items() if k in valid_keys
            }

            # Generate answer using T5 + LoRA
            outputs = model.generate(
                **inputs,
                max_length=64,
                num_beams=4,
                early_stopping=True,
                output_scores=True,
                return_dict_in_generate=True
            )

            # Get generated tokens
            generated_ids = outputs.sequences
            pred_text = tokenizer.decode(generated_ids[0], skip_special_tokens=True).strip()

            # Calculate confidence from transition scores
            if hasattr(outputs, 'scores') and outputs.scores:
                try:
                    transition_scores = model.compute_transition_scores(
                        outputs.sequences, outputs.scores, normalize_logits=True
                    )
                    confidence = torch.exp(transition_scores[0]).mean().item()
                except Exception:
                    confidence = 0.5
            else:
                confidence = 0.5

            confidences.append(confidence)

            ref_text = sample['answer_texts']

            all_preds_text.append(pred_text)
            all_refs_text.append(ref_text)

            # Handle empty predictions for similarity model
            if not pred_text:
                sim_score = 0.0
            else:
                pred_emb = similarity_model.encode(pred_text, convert_to_tensor=True, device='cpu')
                ref_emb = similarity_model.encode(ref_text, convert_to_tensor=True, device='cpu')
                sim_score = util.cos_sim(pred_emb, ref_emb).item()

            similarities.append(sim_score)

    f1_scores = [compute_qa_f1(pred, ref) for pred, ref in zip(all_preds_text, all_refs_text)]
    em_scores = [compute_qa_exact_match(pred, ref) for pred, ref in zip(all_preds_text, all_refs_text)]

    avg_f1 = sum(f1_scores) / len(f1_scores) if f1_scores else 0.0
    avg_em = sum(em_scores) / len(em_scores) if em_scores else 0.0

    bleu_score = bleu_metric.compute(predictions=all_preds_text, references=[[r] for r in all_refs_text])['bleu']
    rouge_score = rouge_metric.compute(predictions=all_preds_text, references=all_refs_text, rouge_types=["rougeL"])['rougeL']
    meteor_score = meteor_metric.compute(predictions=all_preds_text, references=all_refs_text)['meteor']

    avg_confidence = float(np.mean(confidences)) if confidences else 0.0
    avg_similarity = float(np.mean(similarities)) if similarities else 0.0

    return {
        "F1 Score (Token Overlap)": avg_f1,
        "Accuracy (Exact Match)": avg_em,
        "BLEU": bleu_score,
        "ROUGE-L": rouge_score,
        "METEOR": meteor_score,
        "Avg Confidence": avg_confidence,
        "Avg Semantic Similarity": avg_similarity,
        "Training Time (seconds)": training_time
    }


# ✅ Evaluate ONLY on the unseen test set (400 samples instead of 2000)
results = evaluate_model(model, tokenized_test, tokenizer)
print("\n" + "=" * 45)
print("FINAL EVALUATION RESULTS (T5 + LoRA)")
print("=" * 45)
for metric, value in results.items():
    if isinstance(value, float):
        print(f"{metric:<25}: {value:.4f}")
    else:
        print(f"{metric:<25}: {value}")

In [ ]:
import json
import time
import torch
import numpy as np
import matplotlib.pyplot as plt
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    TrainingArguments,
    Trainer,
    DataCollatorForSeq2Seq
)
from peft import AdaLoraConfig, get_peft_model, TaskType
import evaluate
from sentence_transformers import SentenceTransformer, util
import re
import string
from collections import Counter


# ==========================================
# 1. Load and Prepare Data
# ==========================================

json_file = "version1_2_fixed_other.json"   # Change to your file

with open(json_file, "r", encoding="utf-8") as f:
    data = json.load(f)["data"]

flat_data = []
sample_id = 0

for item in data:
    title = item.get("title", "")

    for para in item.get("paragraphs", []):
        context = para.get("context", "")

        for qa in para.get("qas", []):

            # Skip questions without answers
            if not qa.get("answers"):
                continue

            answer = qa["answers"][0]

            flat_data.append({
                "id": qa.get("id", f"sample_{sample_id}"),
                "title": title,
                "context": context,
                "question": qa.get("question", ""),
                "answer_text": answer.get("text", ""),
                "answer_start": answer.get("answer_start", -1),
                "answer_end": answer.get("answer_end", -1)
            })

            sample_id += 1

from sklearn.model_selection import train_test_split

# Split the data: 80% train, 20% test
train_raw, test_raw = train_test_split(flat_data, test_size=0.2, random_state=42)

train_dataset = Dataset.from_list(train_raw)
test_dataset = Dataset.from_list(test_raw)

print(f"✅ Loaded {len(flat_data)} total samples")
print(f"✅ Training samples: {len(train_dataset)}")
print(f"✅ Testing samples: {len(test_dataset)}")
print(train_dataset[0])


# ==========================================
# 2. Tokenization and Preprocessing for T5
# ==========================================
model_checkpoint = "t5-base"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

# T5 requires a prefix for different tasks
prefix = "question: "


def preprocess_function(examples):
    """
    For T5, we format input as: "question: <question> context: <context>"
    and output as just the answer text.
    """
    inputs = [f"{prefix}{q} context: {c}" for q, c in zip(examples["question"], examples["context"])]
    targets = examples["answer_text"]

    model_inputs = tokenizer(
        inputs,
        max_length=384,
        truncation=True,
        padding="max_length"
    )

    labels = tokenizer(
        targets,
        max_length=64,
        truncation=True,
        padding="max_length"
    )

    model_inputs["labels"] = labels["input_ids"]
    model_inputs["example_id"] = examples["id"]
    model_inputs["answer_texts"] = examples["answer_text"]

    return model_inputs


cols_to_remove = ["id", "title", "context", "question", "answer_text", "answer_start", "answer_end"]

# Tokenize train and test SEPARATELY
tokenized_train = train_dataset.map(preprocess_function, batched=True, remove_columns=cols_to_remove)
tokenized_test = test_dataset.map(preprocess_function, batched=True, remove_columns=cols_to_remove)
# ==========================================
# 3. Model Initialization + AdaLoRA + Training
# ==========================================

# 1. Define Training Arguments
training_args = TrainingArguments(
    output_dir="./results_t5",
    eval_strategy="epoch",  # Note: if you get an error about this, change it to evaluation_strategy="epoch"
    learning_rate=2e-4,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    num_train_epochs=3,
    weight_decay=0.01,
    report_to="none",
    use_cpu=True,
    logging_steps=1,
    # predict_with_generate=True,  <-- REMOVED: Not needed for custom evaluation loop
)

# 2. Calculate total training steps for AdaLoRA
# 2. Calculate total training steps for AdaLoRA (Use ONLY training set size!)
num_train_samples = len(tokenized_train)
batch_size = training_args.per_device_train_batch_size
num_epochs = training_args.num_train_epochs
gradient_accumulation_steps = getattr(training_args, 'gradient_accumulation_steps', 1)

# Calculate total steps (ensure it's at least 1 to avoid division errors)
total_steps = max(1, (num_train_samples // (batch_size * gradient_accumulation_steps)) * num_epochs)
print(f"Calculated total training steps: {total_steps}")

# 3. Initialize Base Model
base_model = AutoModelForSeq2SeqLM.from_pretrained(model_checkpoint)

# 4. AdaLoRA configuration for T5
safe_tinit = max(1, total_steps // 4)
safe_tfinal = max(1, total_steps // 4)

adalora_config = AdaLoraConfig(
    task_type=TaskType.SEQ_2_SEQ_LM,
    init_r=8,
    target_r=4,
    beta1=0.85,
    beta2=0.85,
    tinit=safe_tinit,
    tfinal=safe_tfinal,
    deltaT=10,
    lora_alpha=16,
    lora_dropout=0.1,
    target_modules=["q", "v", "k", "o"],  # T5 attention projections
    bias="none",
    total_step=total_steps,
)

model = get_peft_model(base_model, adalora_config)

print("\nAdaLoRA trainable parameters:")
model.print_trainable_parameters()

# Data collator for seq2seq models
data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model)

start_train_time = time.time()

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,   # ✅ Train on the training set
    eval_dataset=tokenized_test,     # ✅ Validate on the unseen test set
    data_collator=data_collator,
)

print("Starting Training...")
trainer.train()

end_train_time = time.time()
training_time = end_train_time - start_train_time
print(f"Training completed in {training_time:.2f} seconds.")


# ==========================================
# 3.5 Plot Training Loss
# ==========================================
log_history = trainer.state.log_history
steps = []
losses = []

for log in log_history:
    if "loss" in log:
        steps.append(log["step"])
        losses.append(log["loss"])

if steps:
    plt.figure(figsize=(10, 5))
    plt.plot(steps, losses, linewidth=2)
    plt.title("Training Loss over Steps (T5)")
    plt.xlabel("Step")
    plt.ylabel("Loss")
    plt.grid(True, alpha=0.3)
    plt.tight_layout()

    # Save the figure
    plt.savefig("training_loss_t5_adalora.png", dpi=300, bbox_inches="tight")
    plt.show()
    plt.close()
else:
    print("No loss history found to plot.")


# ==========================================
# 4. Comprehensive Evaluation Metrics (String-Based QA)
# ==========================================
print("Evaluating Model...")


def normalize_text(s):
    def remove_articles(text):
        return re.sub(r'\b(a|an|the)\b', ' ', text)

    def white_space_fix(text):
        return ' '.join(text.split())

    def remove_punc(text):
        exclude = set(string.punctuation)
        return ''.join(ch for ch in text if ch not in exclude)

    def lower(text):
        return text.lower()

    return white_space_fix(remove_articles(remove_punc(lower(s))))


def compute_qa_f1(prediction, ground_truth):
    prediction_tokens = normalize_text(prediction).split()
    ground_truth_tokens = normalize_text(ground_truth).split()
    common = Counter(prediction_tokens) & Counter(ground_truth_tokens)
    num_same = sum(common.values())
    if num_same == 0:
        return 0.0
    precision = 1.0 * num_same / len(prediction_tokens)
    recall = 1.0 * num_same / len(ground_truth_tokens)
    return 2 * (precision * recall) / (precision + recall)


def compute_qa_exact_match(prediction, ground_truth):
    return 1.0 if normalize_text(prediction) == normalize_text(ground_truth) else 0.0


bleu_metric = evaluate.load("bleu")
rouge_metric = evaluate.load("rouge")
meteor_metric = evaluate.load("meteor")

similarity_model = SentenceTransformer('all-MiniLM-L6-v2', device='cpu')


def evaluate_model(model, dataset, tokenizer):
    device = torch.device("cpu")
    model.to(device)
    model.eval()

    all_preds_text = []
    all_refs_text = []
    confidences = []
    similarities = []

    with torch.no_grad():
        for i in range(len(dataset)):
            sample = dataset[i]

            # Prepare input for generation
            input_ids = torch.tensor(sample['input_ids']).unsqueeze(0).to(device)
            attention_mask = torch.tensor(sample['attention_mask']).unsqueeze(0).to(device)

            # Generate answer using T5
            outputs = model.generate(
                input_ids=input_ids,
                attention_mask=attention_mask,
                max_length=64,
                num_beams=4,
                early_stopping=True,
                output_scores=True,
                return_dict_in_generate=True
            )

            # Get generated tokens
            generated_ids = outputs.sequences[0]
            pred_text = tokenizer.decode(generated_ids, skip_special_tokens=True).strip()

            # Calculate confidence from transition scores if available
            if hasattr(outputs, 'scores') and outputs.scores:
                # Average probability across generated tokens
                transition_scores = outputs.scores
                probs = [torch.softmax(score, dim=-1) for score in transition_scores]
                max_probs = [torch.max(p, dim=-1)[0] for p in probs]
                confidence = torch.prod(torch.stack(max_probs)).item()
            else:
                confidence = 0.5  # Default if no scores available

            confidences.append(confidence)

            ref_text = sample['answer_texts']

            all_preds_text.append(pred_text)
            all_refs_text.append(ref_text)

            # Semantic similarity
            pred_emb = similarity_model.encode(pred_text, convert_to_tensor=True, device='cpu')
            ref_emb = similarity_model.encode(ref_text, convert_to_tensor=True, device='cpu')
            sim_score = util.cos_sim(pred_emb, ref_emb).item()
            similarities.append(sim_score)

    f1_scores = [compute_qa_f1(pred, ref) for pred, ref in zip(all_preds_text, all_refs_text)]
    em_scores = [compute_qa_exact_match(pred, ref) for pred, ref in zip(all_preds_text, all_refs_text)]

    avg_f1 = sum(f1_scores) / len(f1_scores) if f1_scores else 0.0
    avg_em = sum(em_scores) / len(em_scores) if em_scores else 0.0

    bleu_score = bleu_metric.compute(predictions=all_preds_text, references=[[r] for r in all_refs_text])['bleu']
    rouge_score = rouge_metric.compute(predictions=all_preds_text, references=all_refs_text, rouge_types=["rougeL"])['rougeL']
    meteor_score = meteor_metric.compute(predictions=all_preds_text, references=all_refs_text)['meteor']

    avg_confidence = float(np.mean(confidences)) if confidences else 0.0
    avg_similarity = float(np.mean(similarities)) if similarities else 0.0

    return {
        "F1 Score (Token Overlap)": avg_f1,
        "Accuracy (Exact Match)": avg_em,
        "BLEU": bleu_score,
        "ROUGE-L": rouge_score,
        "METEOR": meteor_score,
        "Avg Confidence": avg_confidence,
        "Avg Semantic Similarity": avg_similarity,
        "Training Time (seconds)": training_time
    }


# ✅ Evaluate ONLY on the unseen test set
results = evaluate_model(model, tokenized_test, tokenizer)
print("\n" + "=" * 45)
print("FINAL EVALUATION RESULTS (T5 + AdaLoRA)")
print("=" * 45)
for metric, value in results.items():
    if isinstance(value, float):
        print(f"{metric:<25}: {value:.4f}")
    else:
        print(f"{metric:<25}: {value}")